# Calculate zonal stats on NTL data, masking values to 0 around flares



In [ ]:
import sys
import os
import itertools
import boto3
import rasterio

import geopandas as gpd
import pandas as pd
import numpy as np

from scipy.spatial import cKDTree
from shapely.geometry import Point
from operator import itemgetter
from tqdm.notebook import tqdm

sys.path.insert(0, "../src")

import GOSTrocks.ntlMisc as ntlMisc
import GOSTrocks.rasterMisc as rMisc
from GOSTrocks.misc import tPrint

# Auto-reload modules
%load_ext autoreload
%autoreload 2

In [ ]:
base_folder = "C:/WBG/Work/Projects/NGA"
out_folder = os.path.join(base_folder, "Outputs")
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# I cannot figure out how to download the zonal features from ArcGIS, so I downloaded them manually and saved them as a geojson in the data folder
#zonal_features = "https://services.arcgis.com/iQ1dY19aHwbSDYIF/arcgis/rest/services/Nigeria_Ward_Boundaries_63b55/FeatureServer"

zonal_features = "C:\\WBG\\Work\\Projects\\NGA\\Nigeria_Ward_Boundaries.geojson"
flaring_locations_file = "https://thedocs.worldbank.org/en/doc/d01b4aebd8a10513c0e341de5e1f652e-0400072024/related/2012-2023-individual-flare-volume-estimates.xlsx?_gl=1*19fhic5*_gcl_au*MzM5MTcxNjUwLjE3MTg2NTk5ODU."

# Get a list of all NTL images
s3 = boto3.client('s3', verify=False)
## list all tiff files in an aws prefix
s3_bucket = "wbgdecinternal-ntl"
prefix = "NTL/VIIRS_UNZIP/"

# get a list of all tif files in the prefix
response = s3.list_objects_v2(Bucket=s3_bucket, Prefix=prefix)
# this response is paginated, so we need to loop through all pages to get all files
all_objects = response.get('Contents', [])
while response.get('IsTruncated'):
    response = s3.list_objects_v2(Bucket=s3_bucket, Prefix=prefix, ContinuationToken=response.get('NextContinuationToken'))
    all_objects.extend(response.get('Contents', []))
ntl_images = [f's3://{s3_bucket}/{obj["Key"]}' for obj in all_objects if obj['Key'].endswith('rade9h.tif')]
ntl_images.sort()

zonal_feats = gpd.read_file(zonal_features)

In [ ]:
zonal_res = ntlMisc.run_zonal_flares(zonal_feats, flaring_locations_file, 
                                     ntl_images=ntl_images,buffer_dist=5000, minval=0.1, verbose=True)

In [ ]:
zonal_res

In [ ]:
flares_file = "https://thedocs.worldbank.org/en/doc/d01b4aebd8a10513c0e341de5e1f652e-0400072024/related/2012-2023-individual-flare-volume-estimates.xlsx?_gl=1*19fhic5*_gcl_au*MzM5MTcxNjUwLjE3MTg2NTk5ODU."
flaring_d = pd.read_excel(flares_file)
flaring_d["ID"] = flaring_d.index
flaring_geoms = [Point(x) for x in zip(flaring_d["Longitude"], flaring_d["Latitude"])]
flaring_d = gpd.GeoDataFrame(flaring_d, geometry=flaring_geoms, crs=4326)
buffered_flare = flaring_d.copy().to_crs("ESRI:54009")

buffered_flare["geometry"] = buffered_flare["geometry"].apply(
    lambda x: x.buffer(5000)
)
buffered_flare = buffered_flare.to_crs(4326)

In [ ]:
ntl_image = ntl_images[0]
with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    ntl_r = rasterio.open(ntl_image)
    ntl_window = rasterio.windows.from_bounds(*zonal_feats.total_bounds, transform=ntl_r.transform)
    ntl_data = ntl_r.read(1, window=ntl_window)
    masked_ntl_data = ntl_data.copy()

    temp_meta = ntl_r.meta.copy()
    temp_meta.update({
        "height": ntl_window.height,
        "width": ntl_window.width,
        "transform": rasterio.windows.transform(ntl_window, ntl_r.transform)
    })

In [ ]:
with rMisc.create_rasterio_inmemory(temp_meta, ntl_data) as ntl_raster:
    flare_mask = rMisc.rasterizeDataFrame(buffered_flare, None, templateRaster=ntl_raster, nodata=0)
    flare_mask = (~flare_mask["vals"].astype(bool)).astype(int)
    bool_flare_mask = flare_mask.astype(bool)
    if bool_flare_mask.shape != masked_ntl_data.shape:
        new_mask = np.zeros(masked_ntl_data.shape, dtype=bool)
        new_mask[:bool_flare_mask.shape[0], :bool_flare_mask.shape[1]] = bool_flare_mask
        bool_flare_mask = new_mask
    masked_ntl_data[~bool_flare_mask] = 0